# 5. PDF Onboarding

**Goal:** Enable the agent to answer questions from unstructured PDF documents.

**Key Concept:**
We utilize the Vector DataBase to intelligently parse and chunk the ERCOT market briefing PDF. This enables a RAG (Retrieval Augmented Generation) workflow, allowing the agent to retrieve relevant excerpts and answer questions grounded in the document.

## Create and deploy a DataRobot vector database

This notebook demonstrates how to use the DataRobot Python SDK to create and deploy a vector database (VDB) using DataRobot's built-in embeddings.

## Option-1: Use the already deployed Vector Database

In [ ]:
import os
from pprint import pprint
from dotenv import load_dotenv
import datarobot as dr
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

load_dotenv()

dr_client = dr.Client()

MCP_DEPLOYMENT_ID = os.getenv("MCP_DEPLOYMENT_ID")
if not MCP_DEPLOYMENT_ID:
    raise ValueError("MCP_DEPLOYMENT_ID environment variable is not set")
print(f"Using MCP_DEPLOYMENT_ID={MCP_DEPLOYMENT_ID}")

server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    timeout=60.0,
)

MODEL_NAME = os.getenv("MODEL_NAME", "azure/gpt-5-2025-08-07")
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token,
        base_url=dr_client.endpoint + "/genai/llmgw",
    ),
)

system_prompt = """
You are a Compliance Assistant for the ERCOT Market Briefing.
Use the Vector Database deployment to verify ERCOT Key Metrics.
""".strip()

agent = Agent(model=model, toolsets=[server], system_prompt=system_prompt)

question = "How many customers lost power during Hurricane Francine?"
print(f"\nQuestion: {question}")
print("-" * 30)

async with server:
    response = await agent.run(question)
    pprint(response.output)

## Option-2: Create and Deploy new Vector Database

## Creating a vDB
If you want to go through the whole setup process of the vector database yourself, follow the next steps.

1. Upload the PDF from the documents folder as a ZIP file to the DataRobot AI Catalog.

   **Important:** PDFs cannot be uploaded directly to the DataRobot AI Catalog. They must be zipped first. The code below will automatically:
   - Create a ZIP file containing the PDF
   - Upload the ZIP file to DataRobot AI Catalog
   - Extract the dataset ID for use in vector database creation

In [ ]:
import datarobot as dr
from datarobot.models.genai.vector_database import VectorDatabase
from datarobot.models.genai.vector_database import ChunkingParameters
from datarobot.enums import VectorDatabaseEmbeddingModel
from datarobot.enums import VectorDatabaseChunkingMethod
from datarobot.enums import PredictionEnvironmentPlatform
from datarobot.enums import PredictionEnvironmentModelFormats
import time
import requests

In [ ]:
# Connect to DataRobot
dr.Client()

# Upload PDF to DataRobot AI Catalog (PDFs must be zipped first)
import os
import zipfile
import tempfile

pdf_path = "documents/ercot_market_briefing_enhanced.pdf"

# Check if file exists
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF file not found at: {pdf_path}")

# Create a temporary ZIP file containing the PDF
print(f"Creating ZIP file from {pdf_path}...")
with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmp_zip:
    zip_path = tmp_zip.name
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(pdf_path, os.path.basename(pdf_path))
    print(f"ZIP file created: {zip_path}")

try:
    # Upload the ZIP file to DataRobot AI Catalog
    print(f"Uploading ZIP file to DataRobot AI Catalog...")
    dataset = dr.Dataset.create_from_file(
        file_path=zip_path,
    )
    print(f"Dataset uploaded successfully. Dataset ID: {dataset.id}")
finally:
    # Clean up temporary ZIP file
    if os.path.exists(zip_path):
        os.remove(zip_path)
        print(f"Cleaned up temporary ZIP file: {zip_path}")

In [ ]:
# Create the vector database
chunking_parameters = ChunkingParameters(
    embedding_model=VectorDatabaseEmbeddingModel.JINA_EMBEDDING_T_EN_V1,
    chunking_method=VectorDatabaseChunkingMethod.RECURSIVE,
    chunk_size=256,
    chunk_overlap_percentage=25,
    separators=["\n\n", "\n", " ", ""],
)

2. Create the vector database using the uploaded dataset.

In [ ]:
vdb = VectorDatabase.create(
    dataset_id=dataset.id,  # Use the uploaded dataset
    chunking_parameters=chunking_parameters,
    use_case= os.environ.get('DATAROBOT_USE_CASE_ID')
    name="Vdb_Agent-Build-Clinic_Feb_2026"
)

In [ ]:
max_wait_time = 600
check_interval = 5
start_time = time.time()

print("Waiting for vector database creation...")
while time.time() - start_time < max_wait_time:
    vdb = VectorDatabase.get(vdb.id)
    status = vdb.execution_status
    if status == "COMPLETED":
        print(f"Vector database created: {vdb.name}")
        break
    elif status == "FAILED":
        error_msg = getattr(vdb, 'error_message', 'Unknown error')
        raise Exception(f"Vector database creation failed: {error_msg}")
    else:
        # Show progress if available
        percentage = getattr(vdb, 'percentage', None)
        if percentage is not None:
            print(f"  Status: {status} ({percentage}%)")
        else:
            print(f"  Status: {status}...")
    time.sleep(check_interval)
else:
    raise Exception(f"Vector database creation timed out after {max_wait_time} seconds")

assert vdb.execution_status == "COMPLETED", f"Vector database creation failed with status: {vdb.execution_status}"

In [ ]:
PREDICTION_ENVIRONMENT_NAME = "Vector Database Prediction Environment"

# Get or create prediction environment
prediction_environment = None
for env in dr.PredictionEnvironment.list():
    if env.name == PREDICTION_ENVIRONMENT_NAME:
        prediction_environment = env
        break

if prediction_environment is None:
    prediction_environment = dr.PredictionEnvironment.create(
        name=PREDICTION_ENVIRONMENT_NAME,
        platform=PredictionEnvironmentPlatform.DATAROBOT_SERVERLESS,
        supported_model_formats=[
            PredictionEnvironmentModelFormats.DATAROBOT,
            PredictionEnvironmentModelFormats.CUSTOM_MODEL
        ],
    )
    print(f"Created prediction environment: {prediction_environment.name}")
else:
    print(f"Using existing prediction environment: {prediction_environment.name}")

# Select 3XL resource bundle
resource_bundle_id = None
try:
    dr_client = dr.Client()
    bundles_url = f"{dr_client.endpoint}/mlops/compute/bundles/"
    headers = {"Authorization": f"Bearer {dr_client.token}"}
    bundles_response = requests.get(bundles_url, headers=headers, params={"useCases": "customModel"})
    
    if bundles_response.status_code == 200:
        bundles_data = bundles_response.json()
        if bundles_data.get("data"):
            bundles = bundles_data["data"]
            # Look for 3XL bundle
            bundle_3xl = next((b for b in bundles if "3XL" in b.get("name", "").upper()), None)
            if bundle_3xl:
                resource_bundle_id = bundle_3xl["id"]
                print(f"Selected 3XL bundle: {bundle_3xl['name']}")
            else:
                # Fallback to largest available
                sorted_bundles = sorted(bundles, key=lambda b: b.get("memoryBytes", 0), reverse=True)
                if sorted_bundles:
                    resource_bundle_id = sorted_bundles[0]["id"]
                    print(f"Warning: 3XL bundle not found. Using largest available: {sorted_bundles[0]['name']}")
        else:
            print("Using memory settings (no resource bundles available)")
    else:
        print("Using memory settings (resource bundles not enabled)")
except (ImportError, KeyError) as e:
    print(f"Using memory settings (error checking bundles): {e}")
except requests.RequestException as e:
    print(f"Using memory settings (network error checking bundles): {e}")

In [ ]:
assert vdb.execution_status == "COMPLETED", f"Vector database must be completed. Current status: {vdb.execution_status}"

# Send to workshop with 3XL bundle or memory settings
if resource_bundle_id:
    custom_model_version = vdb.send_to_custom_model_workshop(
        resource_bundle_id=resource_bundle_id,
        replicas=1,
        network_egress_policy=dr.NETWORK_EGRESS_POLICY.PUBLIC,
    )
else:
    custom_model_version = vdb.send_to_custom_model_workshop(
        maximum_memory=4096*1024*1024,
        replicas=1,
        network_egress_policy=dr.NETWORK_EGRESS_POLICY.PUBLIC,
    )

print(f"Custom model version created: {custom_model_version}")

In [ ]:
# Registering model
REGISTERED_MODEL_NAME = f"Vector Database - {vdb.name}"

# Register model (adds new version if model already exists)
existing_models = [m for m in dr.RegisteredModel.list() if m.name == REGISTERED_MODEL_NAME]

if existing_models:
    registered_model_version = dr.RegisteredModelVersion.create_for_custom_model_version(
        custom_model_version_id=custom_model_version.id,
        registered_model_id=existing_models[0].id,
    )
    print(f"Added new version to existing registered model: {REGISTERED_MODEL_NAME}")
else:
    registered_model_version = dr.RegisteredModelVersion.create_for_custom_model_version(
        custom_model_version_id=custom_model_version.id,
        registered_model_name=REGISTERED_MODEL_NAME,
    )
    print(f"Created new registered model: {REGISTERED_MODEL_NAME}")


In [ ]:
registered_model = dr.RegisteredModel.get(registered_model_version.registered_model_id)
max_wait_time = 600
check_interval = 10
start_time = time.time()

print("Waiting for model build to complete...")
while time.time() - start_time < max_wait_time:
    version = registered_model.get_version(registered_model_version.id)
    build_status = getattr(version, 'build_status', None) or getattr(version, 'buildStatus', None)
    
    if build_status in ('READY', 'complete', 'COMPLETE'):
        print(f"Model build completed (status: {build_status})")
        break
    elif build_status in ('FAILED', 'ERROR', 'error'):
        raise Exception(f"Model build failed. Status: {build_status}")
    else:
        print(f"  Build status: {build_status}...")
    time.sleep(check_interval)
else:
    version = registered_model.get_version(registered_model_version.id)
    build_status = getattr(version, 'build_status', None) or getattr(version, 'buildStatus', None)
    raise Exception(f"Model build timed out. Current status: {build_status}")

# Verify ready status
version = registered_model.get_version(registered_model_version.id)
final_status = getattr(version, 'build_status', None) or getattr(version, 'buildStatus', None)
if final_status not in ('READY', 'complete', 'COMPLETE'):
    raise Exception(f"Model not ready for deployment. Status: {final_status}")


In [ ]:

deployment = dr.Deployment.create_from_registered_model_version(
    registered_model_version.id,
    label=f"Vector Database Deployment - {vdb.name}",
    description="Vector database deployment for RAG applications",
    prediction_environment_id=prediction_environment.id,
    max_wait=600,
)

print(f"Deployment created: {deployment.id}")


Now MCP server will have access of this newly deployed Vector Database and we can leverage that context

In [0]:
import os
MCP_DEPLOYMENT_ID = os.getenv("MCP_DEPLOYMENT_ID")

import datarobot as dr
from pprint import pprint
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

# 1. Setup MCP
dr_client = dr.Client()
server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    timeout=60.0,
)

# 2. Configure Model
model = OpenAIChatModel(
    "azure/gpt-5-2025-08-07", 
    provider=OpenAIProvider(
        api_key=dr_client.token, 
        base_url=f"{dr_client.endpoint}/genai/llmgw"
    ),
)

# 3. Define Agent
system_prompt = "You are a Compliance Assistant for the ERCOT Market Briefing. Use the Vector Database deployment  to verify ERCOT Key Metrics."
agent = Agent(model=model, toolsets=[server], system_prompt=system_prompt)

# 4. Run Query
async def run_query():
    async with server:
        question = "How much economic curtailment occurred in September 2025?"
        print(f"\nQuestion: {question}")
        print("-" * 30)
        response = await agent.run(question)
        pprint(response.output)

await run_query()